In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install torch torchvision torchaudio
!pip install numpy pandas pillow tqdm


In [ ]:
!pip install py-feat==0.4.0
!pip install opencv-python
!pip install scikit-learn


  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray 2025.7.1 requires numpy>=1.26, but you have numpy 1.23.5 which is incompatible.
arviz 0.22.0 requires numpy>=1.26.0, but you have numpy 1.23.5 which is incompatible.
arviz 0.22.0 requires scipy>=1.11.0, but you have scipy 1.10.1 which is incompatible.
jaxlib 0.5.3 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
jaxlib 0.5.3 requires scipy>=1.11.1, but you have scipy 1.10.1 which is incompatible.
albucore 0.0.24 requires numpy>=1.24.4, but y

  Using cached numpy-2.2.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
^C


In [1]:
!pip install scipy==1.10.1


In [2]:
!pip install einops


In [3]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [4]:

# Emotion categories as per your mapping
EMOTION_MAP = {
    'happiness': 0,
    'disgust': 1,
    'repression': 2,
    'surprise': 3,
    'sadness': 4,
    'others': 5
}

# Image transformation for frames
frame_transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0,1] tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

In [9]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from glob import glob

class MicroExpressionDataset(Dataset):
    def __init__(self, root_dir, seq_len=16, transform=None, cache_dir=None):
        print("📂 Initializing MicroExpressionDataset...")
        self.root_dir = root_dir
        self.seq_len = seq_len
        self.transform = transform

        # Force cache directory to your specified path
        self.cache_dir = "/content/drive/MyDrive/deepfake-detection/microexpression/cache"
        os.makedirs(self.cache_dir, exist_ok=True)
        print(f"🗂 Cache directory set to: {self.cache_dir}")

        self.samples = []
        self.label_map = {
            "happiness": 0,
            "disgust": 1,
            "repression": 2,
            "surprise": 3,
            "sadness": 4,
            "others": 5
        }

        print("🔍 Scanning dataset folders...")
        for emotion in os.listdir(root_dir):
            emotion_dir = os.path.join(root_dir, emotion)
            if not os.path.isdir(emotion_dir):
                continue
            for video in os.listdir(emotion_dir):
                video_dir = os.path.join(emotion_dir, video)
                if os.path.isdir(video_dir):
                    self.samples.append((video_dir, self.label_map[emotion]))
        print(f"✅ Found {len(self.samples)} video samples across {len(self.label_map)} emotion categories.")

    def __len__(self):
        return len(self.samples)

    def _load_from_cache(self, cache_path):
        print(f"⚡ Loading from cache: {os.path.basename(cache_path)}")
        return torch.load(cache_path)

    def _save_to_cache(self, cache_path, data):
        torch.save(data, cache_path)
        print(f"💾 Saved to cache: {os.path.basename(cache_path)}")

    def __getitem__(self, idx):
        video_dir, label = self.samples[idx]
        cache_path = os.path.join(self.cache_dir, f"{os.path.basename(video_dir)}.pt")

        if os.path.exists(cache_path):
            return self._load_from_cache(cache_path)

        print(f"\n🎞 Processing video: {os.path.basename(video_dir)} | Emotion: {label}")

        # Load frames
        frame_files = sorted(glob(os.path.join(video_dir, "*.jpg")))
        print(f"🖼 Found {len(frame_files)} frames, loading...")
        frames = []
        for f in frame_files[:self.seq_len]:
            img = Image.open(f).convert("RGB")
            if self.transform:
                img = self.transform(img)
            else:
                img = torch.tensor(np.array(img)).permute(2, 0, 1).float() / 255.0
            frames.append(img)

        while len(frames) < self.seq_len:
            frames.append(torch.zeros_like(frames[0]))
        frames = torch.stack(frames)

        # Load AUs
        au_path = os.path.join(video_dir, "AUs.npy")
        if os.path.exists(au_path):
            aus = torch.tensor(np.load(au_path), dtype=torch.float32)
            if aus.shape[0] > self.seq_len:
                aus = aus[:self.seq_len]
            elif aus.shape[0] < self.seq_len:
                pad_len = self.seq_len - aus.shape[0]
                aus = torch.cat([aus, torch.zeros(pad_len, aus.shape[1])], dim=0)
            print(f"📊 Loaded AUs: {aus.shape}")
        else:
            print("⚠️ No AUs found, filling with zeros.")
            aus = torch.zeros(self.seq_len, 12)

        label_tensor = torch.tensor(label, dtype=torch.long)
        sample = (frames, aus, label_tensor)

        self._save_to_cache(cache_path, sample)
        return sample


if __name__ == "__main__":
    dataset = MicroExpressionDataset(
        root_dir="/content/drive/MyDrive/deepfake-detection/dataset/microexpression_processed",
        seq_len=16,
        transform=None
    )
    print(f"\n📦 Dataset ready. Total samples: {len(dataset)}")
    frames, aus, label = dataset[0]
    print("✅ Sample loaded")
    print("Frames:", frames.shape)
    print("AUs:", aus.shape)
    print("Label:", label)


📂 Initializing MicroExpressionDataset...
🗂 Cache directory set to: /content/drive/MyDrive/deepfake-detection/microexpression/cache
🔍 Scanning dataset folders...
✅ Found 220 video samples across 6 emotion categories.

📦 Dataset ready. Total samples: 220
⚡ Loading from cache: EP02_01f.pt
✅ Sample loaded
Frames: torch.Size([16, 3, 224, 224])
AUs: torch.Size([16, 12])
Label: tensor(0)
